In [1]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import copy
import gymnasium as gym
from tqdm import tqdm
import sys
sys.path.insert(0, "../src/")
from FwdNeuron import *
from DeepEligNeuron import *
from Network import *
from inputFuc import *
from utils import *
from plotting import *
from utilsRL import *

In [2]:
data_config = dict(default_data_config)
general_config = dict(default_general_config)
train_config = dict(default_train_config)


#### With P learning ###
dt = general_config["dt"]

# init both teacher and student net
batch = train_config["batch_size"]
general_config["device"] = "cuda" if torch.cuda.is_available() else "cpu"
device = general_config["device"]
print(device)

env = CartPoleCustomize()
#envs = gym.make_vec("CartPole-v1", num_envs=3, vectorization_mode="vector_entry_point")

n_actions = env.env.action_space.n.item()
# Get the number of state observations
obs, info = env.reset()
n_observations = obs.shape[-1]

cpu


In [33]:
actor_config = dict(default_model_config)
critic_config = dict(default_model_config)
actor_config['n_in'] = n_observations
critic_config['n_in'] = n_observations
actor_config['n_out'] = n_actions
critic_config['n_out'] = 1

actor = buildRLNet(actor_config, general_config).to(device)
critic = buildRLNet(critic_config, general_config).to(device)
critic_target = copy.deepcopy(critic)

num_episode = 10000

optimizer_actor = torch.optim.Adam(
    actor.parameters(),
    lr=2e-4,#train_config["learning_rate"],
    betas=(0.95, 0.999)
)

optimizer_critic = torch.optim.Adam(
    critic.parameters(),
    lr=5e-3,#train_config["learning_rate"],
    betas=(0.95, 0.999)
)

scheduler_actor = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_actor,
    T_max = num_episode,
)

scheduler_critic = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_critic,
    T_max = num_episode,
)

error_record = []
Cum_errors = 0.5

tensor([1.1000, 2.4000, 2.0000, 3.0000])
tensor([1.1000, 2.4000, 2.0000, 3.0000])


In [34]:
def episode(env, actor, critic, critic_target, optimizer_actor, optimizer_critic):
    observation, _ = env.reset()
    
    actor.reset()
    critic.reset()
    critic_target.reset()
    gamma=0.9
    # Run one episode
    terminated, truncated = False, False
    total_reward=0.
    with torch.no_grad():
        while not terminated and not truncated:
            # Render the environment
            #env.render()
            # Take a random action
            u, _ = actor.step(observation)
            p = torch.softmax(u, dim=1)
            action = np.random.choice(n_actions, p=p[0].cpu().numpy())
            value,_ = critic.step(observation)
            # Step the environment
            observation, reward, terminated, truncated,_ = env.step(action)
            value_next,_ = critic_target.step(observation)
            
            delta = reward + gamma*value_next - value
            
            optimizer_actor.zero_grad(set_to_none=False)
            optimizer_critic.zero_grad(set_to_none=False)
            
            actor.prop(-delta*(F.one_hot(torch.tensor(action), num_classes=n_actions)-p))
            critic.prop(delta)
            actor.backwards()
            critic.backwards()
            # actor.prop(1-p)
            # actor.backwardsRL(-delta)
            # critic.prop(1.)
            # critic.backwardsRL(delta)
            
            optimizer_actor.step()
            optimizer_critic.step()
            total_reward+=reward
        critic_target = copy.deepcopy(critic)

    return total_reward

In [35]:
# Create the CartPole environment

R = 10
for i in range(num_episode):
    total_reward = episode(env, actor, critic, critic_target, optimizer_actor, optimizer_critic)
    scheduler_actor.step()
    scheduler_critic.step()
    R = 0.999*R + 0.001*total_reward
    if i%500==0:
        print(R)
# Close the environment
env.close()
print("finished")

9.014797227436313
10.450362941684569
10.849504163724976
10.827794626824142
12.584827914672804
12.283318950967566
13.541921000897988
13.088524417765592
13.697171600726309
15.6090870774514
16.999295404837724
17.556842783621086
18.86358540970844
19.13523233271481
18.947255343456355
17.281081981349196
16.455575741757382
16.578639591550886
16.735627001901694
16.606533246206723
finished
